# Lab 4 Extension: DRAM Weight Streaming Metadata in MASE

This notebook shows how to configure MASE hardware metadata so layer parameters are streamed from DRAM instead of generated as BRAM ROMs.

Goals:
- Build and trace a small model into a `MaseGraph`
- Compare `storage="BRAM"` vs `storage="DRAM"` metadata
- Validate metadata programmatically
- Run Verilog emit passes and check streaming ports in `top.sv`

In [1]:
from __future__ import annotations

from copy import deepcopy
from pathlib import Path

import torch
import torch.nn as nn

from chop import MaseGraph
from chop.passes.graph.analysis import (
    add_common_metadata_analysis_pass,
    add_hardware_metadata_analysis_pass,
    init_metadata_analysis_pass,
)
from chop.passes.graph.transforms import (
    emit_bram_transform_pass,
    emit_cocotb_transform_pass,
    emit_internal_rtl_transform_pass,
    emit_verilog_top_transform_pass,
    quantize_transform_pass,
)

torch.manual_seed(0)

print(f"Torch version: {torch.__version__}")

/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch version: 2.6.0+cu124


In [2]:
class StreamReadyMLP(nn.Module):
    """Small MLP used to demonstrate parameter storage metadata."""

    def __init__(self) -> None:
        super().__init__()
        self.fc1 = nn.Linear(4, 8)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(8, 4)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.flatten(x, start_dim=1, end_dim=-1)
        x = self.relu(self.fc1(x))
        return self.fc2(x)


def build_quantized_graph(model: nn.Module, dummy_x: torch.Tensor) -> MaseGraph:
    """Trace model, add common metadata, and quantize to fixed-point."""
    mg = MaseGraph(model=model)
    dummy_in = {"x": dummy_x}

    mg, _ = init_metadata_analysis_pass(mg)
    mg, _ = add_common_metadata_analysis_pass(
        mg,
        {"dummy_in": dummy_in, "add_value": False, "force_device_meta": False},
    )

    quant_config = {
        "by": "type",
        "default": {"config": {"name": None}},
        "linear": {
            "config": {
                "name": "integer",
                "data_in_width": 8,
                "data_in_frac_width": 3,
                "weight_width": 8,
                "weight_frac_width": 3,
                "bias_width": 8,
                "bias_frac_width": 3,
            }
        },
        "relu": {"config": {"name": None}},
    }
    mg, _ = quantize_transform_pass(mg, pass_args=quant_config)
    return mg

In [7]:
def _node_parameters(node):
    """Return normalized Mase parameter dict for a node."""
    mase_meta = node.meta.get("mase")
    if mase_meta is None:
        return {}
    if hasattr(mase_meta, "parameters"):
        return mase_meta.parameters
    if isinstance(mase_meta, dict):
        return mase_meta.get("parameters", mase_meta)
    return {}


def collect_parameter_storage(mg: MaseGraph) -> dict[str, str]:
    """Return storage mode for each non-input tensor argument in the graph."""
    out = {}
    for node in mg.fx_graph.nodes:
        params = _node_parameters(node)
        if params.get("hardware", {}).get("is_implicit", False):
            continue

        args_meta = params.get("common", {}).get("args", {})
        iface_meta = params.get("hardware", {}).get("interface", {})

        for arg_name, arg_info in args_meta.items():
            if "data_in" in arg_name or not isinstance(arg_info, dict):
                continue
            storage = iface_meta.get(arg_name, {}).get("storage", "<missing>")
            out[f"{node.name}.{arg_name}"] = storage
    return out


def apply_hardware_metadata(mg: MaseGraph, storage_mode: str) -> MaseGraph:
    """Apply hardware metadata with explicit interface storage mode."""
    mg, _ = add_hardware_metadata_analysis_pass(
        mg,
        {"interface": {"storage": storage_mode}},
    )
    return mg


def assert_storage_mode(mg: MaseGraph, expected: str) -> None:
    storages = collect_parameter_storage(mg)
    assert storages, "No parameter storage metadata found in graph."
    bad = {k: v for k, v in storages.items() if v != expected}
    assert not bad, f"Storage mismatch. Expected '{expected}', got {bad}"

In [8]:
dummy_x = torch.randn(1, 2, 2)

# Build independent graphs to avoid any cross-mutation between metadata variants.
torch.manual_seed(0)
mg_bram = build_quantized_graph(StreamReadyMLP().eval(), dummy_x)
mg_bram = apply_hardware_metadata(mg_bram, "BRAM")

torch.manual_seed(0)
mg_dram = build_quantized_graph(StreamReadyMLP().eval(), dummy_x)
mg_dram = apply_hardware_metadata(mg_dram, "DRAM")

assert_storage_mode(mg_bram, "BRAM")
assert_storage_mode(mg_dram, "DRAM")

bram_storage = collect_parameter_storage(mg_bram)
dram_storage = collect_parameter_storage(mg_dram)

print("BRAM metadata entries:")
for name in sorted(bram_storage):
    print(f"  {name}: {bram_storage[name]}")

print("\nDRAM metadata entries:")
for name in sorted(dram_storage):
    print(f"  {name}: {dram_storage[name]}")

BRAM metadata entries:
  fc1.bias: BRAM
  fc1.weight: BRAM
  fc2.bias: BRAM
  fc2.weight: BRAM

DRAM metadata entries:
  fc1.bias: DRAM
  fc1.weight: DRAM
  fc2.bias: DRAM
  fc2.weight: DRAM


## Emit RTL for DRAM Streaming Mode

The pass sequence is unchanged; only metadata changes.
In DRAM mode, parameter BRAM sources are skipped and top-level streaming handshake ports are emitted for weights/biases.

In [11]:
project_dir = (Path.cwd() / "build" / "lab4_dram_streaming").resolve()
emit_args = {"project_dir": project_dir, "top_name": "top"}

mg_dram_emit = deepcopy(mg_dram)
mg_dram_emit, _ = emit_verilog_top_transform_pass(mg_dram_emit, pass_args=emit_args)
mg_dram_emit, _ = emit_internal_rtl_transform_pass(mg_dram_emit, pass_args=emit_args)
mg_dram_emit, _ = emit_bram_transform_pass(mg_dram_emit, pass_args=emit_args)

# Cocotb testbench emission can fail in some environments due GraphModule serialization.
try:
    mg_dram_emit, _ = emit_cocotb_transform_pass(mg_dram_emit, pass_args=emit_args)
    cocotb_status = "ok"
except Exception as exc:
    cocotb_status = f"skipped ({type(exc).__name__}: {exc})"
    print(f"Warning: emit_cocotb_transform_pass failed; continuing. {cocotb_status}")

rtl_top = project_dir / "hardware" / "rtl" / "top.sv"
assert rtl_top.exists(), f"Expected generated RTL file: {rtl_top}"

print(f"Generated project directory: {project_dir}")
print(f"Generated top module: {rtl_top}")
print(f"Cocotb emit status: {cocotb_status}")

Generated project directory: /home/ism/ADL/mase/docs/labs/build/lab4_dram_streaming
Generated top module: /home/ism/ADL/mase/docs/labs/build/lab4_dram_streaming/hardware/rtl/top.sv
Cocotb emit status: skipped (AttributeError: 'str' object has no attribute '__name__')


In [10]:
rtl_top = (Path.cwd() / "build" / "lab4_dram_streaming" / "hardware" / "rtl" / "top.sv").resolve()
top_text = rtl_top.read_text()

# Quick sanity checks for DRAM-style streaming ports in the top-level module.
expected_tokens = ["_weight", "_weight_valid", "_weight_ready", "_bias", "_bias_valid", "_bias_ready"]
missing = [tok for tok in expected_tokens if tok not in top_text]
assert not missing, f"Did not find expected DRAM streaming tokens in top.sv: {missing}"

print("Found expected DRAM parameter streaming signal tokens in top.sv")

Found expected DRAM parameter streaming signal tokens in top.sv


## Notes

- Use `storage="BRAM"` for default on-chip parameter ROM generation.
- Use `storage="DRAM"` to expose parameter streams at `top.sv` and drive them externally (or by cocotb drivers).
- If you have older scripts using `storage=""`, migrate to `storage="DRAM"` for clarity and alignment with current testbench logic.